# DeepSeek 7B + Anthropic HH Steering Vector on Kaggle GPU

Fresh notebook for computing and applying a steering vector with a DeepSeek 7B chat model on Kaggle.

Pipeline:

`HH dataset -> chosen/rejected pairs -> hidden states -> steering vector -> safe hook -> evaluation`


## 1. Kaggle runtime checklist

Before running:

- Enable `Internet`
- Set `Accelerator` to `GPU`
- Add a Kaggle secret named `HF_TOKEN` for authenticated Hugging Face downloads
- This notebook is designed for a 7B chat model loaded in full 16-bit precision on Kaggle GPU

In [1]:
!pip -q install -U transformers datasets accelerate sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 71.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.2 MB/s eta 0:00:00:00:01


## 2. Imports and runtime config

In [2]:
import os
from pathlib import Path

import torch
from datasets import load_dataset
from huggingface_hub import login
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

os.environ["TOKENIZERS_PARALLELISM"] = "false"
set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

if DEVICE == "cuda" and torch.cuda.is_bf16_supported():
    COMPUTE_DTYPE = torch.bfloat16
elif DEVICE == "cuda":
    COMPUTE_DTYPE = torch.float16
else:
    COMPUTE_DTYPE = torch.float32

print(f"Device: {DEVICE}")
print(f"Compute dtype: {COMPUTE_DTYPE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")

# Silence the harmless readonly history warning in some Kaggle sessions.
try:
    from IPython import get_ipython
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic("config", "HistoryManager.enabled = False")
except Exception:
    pass

Device: cuda
Compute dtype: torch.bfloat16
GPU: Tesla T4
Output dir: /kaggle/working


## 3. Optional Hugging Face login

In [3]:
HF_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("Loaded HF token from Kaggle secrets.")
except Exception:
    print("Kaggle secret 'HF_TOKEN' not found.")

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("Loaded HF token from environment.")

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face login successful.")
else:
    print("Proceeding without HF login. Public models may still download.")

Loaded HF token from Kaggle secrets.
Hugging Face login successful.


## 4. What went wrong in the previous run

The earlier notebook hit a few real issues:

1. `chosen` and `rejected` in HH-RLHF already contain the conversation content, so using a separate `prompt` field was incorrect.
2. A forward hook stayed attached after reruns, which caused stale behavior across notebook cells.
3. With `device_map="auto"`, the hook tensor was sometimes on a different GPU device than the decoder layer output.
4. The chat model was prompted like a plain completion model, which weakens output quality.
5. The `max_new_tokens` vs `max_length` warning came from the generation config, not from the experiment logic.

This fresh notebook avoids those issues from the start.

## 5. Load DeepSeek 7B chat model

This notebook assumes the 7B chat checkpoint `deepseek-ai/deepseek-llm-7b-chat`.

This version keeps the model in full GPU precision instead of quantizing it.

In [23]:
MODEL_NAME = "deepseek-ai/deepseek-llm-7b-chat"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


model_kwargs = {
    "device_map": "auto" if DEVICE == "cuda" else None,
    "low_cpu_mem_usage": True
}

if DEVICE == "cuda":
    model_kwargs["dtype"] = COMPUTE_DTYPE
else:
    model_kwargs["dtype"] = torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.eval()

if hasattr(model, "generation_config") and hasattr(model.generation_config, "max_length"):
    model.generation_config.max_length = None

print(type(model).__name__)
print(f"Layers: {len(model.model.layers)}")

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

LlamaForCausalLM
Layers: 30


## 6. Load Anthropic HH-RLHF dataset

In [5]:
NUM_SAMPLES = 300
dataset = load_dataset("Anthropic/hh-rlhf", split=f"train[:{NUM_SAMPLES}]")
print(dataset)
print(dataset[0].keys())

README.md: 0.00B [00:00, ?B/s]

harmless-base/train.jsonl.gz:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

helpful-base/train.jsonl.gz:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

helpful-online/train.jsonl.gz:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

helpful-rejection-sampled/train.jsonl.gz:   0%|          | 0.00/25.7M [00:00<?, ?B/s]

...

helpful-online/test.jsonl.gz:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

helpful-rejection-sampled/test.jsonl.gz:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/160800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8552 [00:00<?, ? examples/s]

Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 300
})
dict_keys(['chosen', 'rejected'])


## 7. Build contrastive pairs

Use `chosen` as positive and `rejected` as negative.

In [6]:
pairs = []

for row in dataset:
    pairs.append((row["chosen"], row["rejected"]))

print(f"Built {len(pairs)} pairs")
print("\nPositive preview:\n")
print(pairs[0][0][:800])
print("\nNegative preview:\n")
print(pairs[0][1][:800])

Built 300 pairs

Positive preview:



Human: What are some cuss words in english?

Assistant: Here’s an incomplete list.

Ass, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cumbucket, fucknugget, butthole, poop, jackass, cocksucker, asshole, goddamn, piss, sperm, blow, wank, jism, cum-sucking, masturbate, faggot, queer, jizz, jizz-licking, prostitute, slut, cheater, fornicator, floozy, wetback, Mexican, Hispanic, sodomite, midget, mama’s boy, faggot, pervert, queer, scumbag, bitch,

Human: What's your favorite one?

Assistant: I haven't even thought about it.

Negative preview:



Human: What are some cuss words in english?

Assistant: Here’s an incomplete list.

Ass, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cumbucket, fucknugget, butthole, poop, jackass, cocksucker, asshole, goddamn, piss, sperm, blow, wank, jism, cum-sucking, masturbate, faggot, que

## 8. Choose a steering layer

We start in the upper-middle of the stack and clamp to the model depth.

In [7]:
LAYER = min(20, len(model.model.layers) - 1)
MAX_LENGTH = 512
print(f"Using layer {LAYER}")

Using layer 20


## 9. Hidden-state extraction helpers

In [8]:
def move_to_model_device(batch):
    target_device = next(model.parameters()).device
    return {k: v.to(target_device) for k, v in batch.items()}

def get_hidden(text, layer=LAYER, max_length=MAX_LENGTH):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )
    inputs = move_to_model_device(inputs)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    hidden = outputs.hidden_states[layer]
    return hidden[:, -1, :].squeeze(0).detach().float().cpu()

## 10. Compute the steering vector

In [9]:
pos_vecs = []
neg_vecs = []

for pos, neg in tqdm(pairs):
    pos_vecs.append(get_hidden(pos))
    neg_vecs.append(get_hidden(neg))

pos_mean = torch.stack(pos_vecs).mean(dim=0)
neg_mean = torch.stack(neg_vecs).mean(dim=0)

v = pos_mean - neg_mean
v = v / v.norm()

print("Vector shape:", tuple(v.shape))
print("Vector norm:", v.norm().item())

  0%|          | 0/300 [00:00<?, ?it/s]

Vector shape: (4096,)
Vector norm: 0.9999998211860657


## 11. Sanity-check the vector

In [10]:
def score(text):
    h = get_hidden(text)
    return torch.dot(h, v).item()

pos_score = score(pairs[0][0])
neg_score = score(pairs[0][1])
print("Positive score:", pos_score)
print("Negative score:", neg_score)
print("Positive > Negative:", pos_score > neg_score)

Positive score: -2.154400110244751
Negative score: -8.176530838012695
Positive > Negative: True


## 12. Save the vector

In [11]:
vector_path = OUTPUT_DIR / "deepseek_7b_hh_steering_vector.pt"
torch.save({"layer": LAYER, "vector": v}, vector_path)
print(f"Saved to {vector_path}")

Saved to /kaggle/working/deepseek_7b_hh_steering_vector.pt


## 13. Register a safe steering hook

This version avoids stale hooks and only steers the last token position to reduce generation collapse.

In [12]:
alpha = 0.5
handle = None

def clear_layer_hooks(layer_index=LAYER):
    model.model.layers[layer_index]._forward_hooks.clear()
    model.model.layers[layer_index]._forward_pre_hooks.clear()

def make_steering_hook(vector, alpha_value):
    def hook_fn(module, inputs, output):
        if isinstance(output, tuple):
            hidden_states = output[0]
            steering = vector.to(device=hidden_states.device, dtype=hidden_states.dtype)
            hidden_states[:, -1, :] = hidden_states[:, -1, :] + alpha_value * steering.view(1, -1)
            return (hidden_states,) + output[1:]

        steering = vector.to(device=output.device, dtype=output.dtype)
        output[:, -1, :] = output[:, -1, :] + alpha_value * steering.view(1, -1)
        return output

    return hook_fn

def remove_steering_hook():
    global handle
    if handle is not None:
        handle.remove()
        handle = None
    clear_layer_hooks(LAYER)

def add_steering_hook(alpha_value=alpha):
    global handle
    remove_steering_hook()
    handle = model.model.layers[LAYER].register_forward_hook(
        make_steering_hook(v, alpha_value)
    )

print(f"Prepared steering hook at layer {LAYER} with alpha={alpha}")

Prepared steering hook at layer 20 with alpha=0.5


## 14. Generation helper with chat formatting

In [24]:
def format_chat_prompt(prompt):
    messages = [{"role": "user", "content": prompt}]
    device = next(model.parameters()).device

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        )

        if isinstance(formatted, torch.Tensor):
            input_ids = formatted.to(device)
            attention_mask = torch.ones_like(input_ids)
            return input_ids, attention_mask

        if isinstance(formatted, dict) or hasattr(formatted, "keys"):
            formatted = {k: v.to(device) for k, v in formatted.items()}
            input_ids = formatted["input_ids"]
            attention_mask = formatted.get("attention_mask", torch.ones_like(input_ids))
            return input_ids, attention_mask

    text = f"User: {prompt}\nAssistant:"
    encoded = tokenizer(text, return_tensors="pt")
    encoded = {k: v.to(device) for k, v in encoded.items()}
    return encoded["input_ids"], encoded.get("attention_mask", torch.ones_like(encoded["input_ids"]))
def generate(prompt, max_new_tokens=100, temperature=0.7, top_p=0.9):
    input_ids, attention_mask = format_chat_prompt(prompt)

    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = output[0][input_ids.shape[-1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    text = text.replace("Ġ", " ").replace("▁", " ")
    text = " ".join(text.split())

    return text.strip()


## 15. Compare base vs steered generation

In [25]:
prompt = "Explain quantum computing in simple terms."

remove_steering_hook()
base_text = generate(prompt)

add_steering_hook(alpha)
steered_text = generate(prompt)

remove_steering_hook()

print("BASE:\n")
print(base_text)
print("\n" + "=" * 80 + "\n")
print("STEERED:\n")
print(steered_text)

BASE:

Quantum computing is a type of computing that uses quantum-mechanical phenomena, such as superposition and entanglement, to perform operations on data. In a classical computer, data is stored in bits, which can have a value of either 0 or 1. In a quantum computer, data is stored in quantum bits, or qubits, which can be in multiple states simultaneously. This allows a quantum computer to perform many operations at the same time, potentially making it much faster and more powerful than a classical computer.


STEERED:

Quantum computing is a type of computing that uses quantum-mechanical phenomena, such as superposition and entanglement, to perform operations on data. In classical computing, data is stored in bits, which can have a value of either 0 or 1. In quantum computing, data is stored in quantum bits, or qubits, which can have a value of 0, 1, or both at the same time. This allows quantum computers to perform certain types of calculations much faster than classical computer

## 16. Small sweep for alpha and prompt quality

Start with low steering strength. This is more stable than jumping straight to `alpha=2.0`.

In [26]:
test_prompts = [
    "Explain quantum computing in simple terms.",
    "How should I politely disagree with a coworker?",
    "Write a short, helpful answer about healthy sleep habits."
]

for alpha_try in [0.25, 0.5, 1.0]:
    print("\n" + "#" * 90)
    print(f"ALPHA = {alpha_try}")

    for prompt in test_prompts:
        print("\nPROMPT:", prompt)
        remove_steering_hook()
        base_text = generate(prompt, max_new_tokens=80)

        add_steering_hook(alpha_try)
        steered_text = generate(prompt, max_new_tokens=80)
        remove_steering_hook()

        print("\nBASE:\n")
        print(base_text)
        print("\nSTEERED:\n")
        print(steered_text)



##########################################################################################
ALPHA = 0.25

PROMPT: Explain quantum computing in simple terms.

BASE:

Quantum computing is a type of computing that uses the principles of quantum mechanics to perform operations on data. In classical computing, data is represented as bits, which can have a value of either 0 or 1. In quantum computing, data is represented as quantum bits, or qubits, which can exist in multiple states at the same time. This allows quantum computers to perform certain calculations much faster than classical

STEERED:

Quantum computing is a type of computing that uses quantum-mechanical phenomena, such as superposition and entanglement, to perform operations on data. In classical computing, data is represented using bits that can have a value of either 0 or 1. In quantum computing, data is represented using quantum bits, or qubits, which can have a value of 0, 1, or both at the same time

PROMPT: How should I p